# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model was already trained under `GroupKFold` grouped by `client_hash_id` — the honest split. To make the “before/after” comparison this assignment asks for, the "before" here is a **naive random split** (rows shuffled and split with no regard for client boundaries), which is the split someone would default to without thinking about client-level leakage. If pages from the same client share client-level patterns, a random split lets that shared pattern leak between train and test through client identity rather than genuine feature signal — which should show up as an *inflated*, less trustworthy score.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

page_level_full = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) as ctr_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id, AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

page_level_full = page_level_full[page_level_full["avg_position_h1"] > 0].copy()
page_level_full["needs_refresh"] = (page_level_full["avg_position_h2"] > page_level_full["avg_position_h1"]).astype(int)

feature_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
X = page_level_full[feature_cols]
y = page_level_full["needs_refresh"]
groups = page_level_full["client_hash_id"]

# --- BEFORE: naive random split (no client grouping) ---
naive_scores = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, te_idx in kf.split(X):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    naive_scores.append(roc_auc_score(y.iloc[te_idx], rf.predict_proba(X.iloc[te_idx])[:, 1]))

# --- AFTER: honest split, grouped by client_hash_id ---
honest_scores = []
gkf = GroupKFold(n_splits=5)
for tr_idx, te_idx in gkf.split(X, groups=groups):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    honest_scores.append(roc_auc_score(y.iloc[te_idx], rf.predict_proba(X.iloc[te_idx])[:, 1]))

print(f"BEFORE (naive random split):   mean AUC = {np.mean(naive_scores):.4f}  (std={np.std(naive_scores):.4f})")
print(f"AFTER  (grouped by client):    mean AUC = {np.mean(honest_scores):.4f}  (std={np.std(honest_scores):.4f})")
print(f"\nDifference: {np.mean(naive_scores) - np.mean(honest_scores):+.4f}")
print("If the naive split scores meaningfully higher, that gap is an estimate of how much")
print("client-level leakage was inflating the number - not real predictive power.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Week 3's hunt was on the starter-sample feature set (`trend_pct`, `ctr`, `avg_position` from `content_refresh_anonymized.csv`). The Week-5 model uses a **different** feature set on the warehouse data (`avg_impressions_h1`, `avg_clicks_h1`, `ctr_h1`, `avg_position_h1`, `active_days_h1`), so the hunt has to be redone against these actual columns, not assumed safe by association with the earlier, already-solved audit.

Same method as Week 3: attack the features directly rather than argue from design intent. The deliberate test below adds the outcome-window value itself as a feature and checks whether the validation setup is sensitive enough to catch it.

In [ ]:
# Check 1: every feature name ends in _h1 (first-half only) by construction - confirm
# none of them are silently pulling h2 (second-half / outcome-window) data.
print("Feature columns:", feature_cols)
print("All end in '_h1':", all(c.endswith("_h1") for c in feature_cols))

# Check 2: deliberate leakage test - add the outcome-window value as a feature on purpose,
# and confirm the score becomes suspiciously close to perfect. If it doesn't move at all,
# that's a sign the validation setup wouldn't catch a real accidental leak either.
from sklearn.model_selection import GroupKFold

leaky_features = feature_cols + ["avg_position_h2"]  # the actual outcome-window column
Xl = page_level_full[leaky_features]

leaky_scores = []
for tr_idx, te_idx in gkf.split(Xl, groups=groups):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xl.iloc[tr_idx], y.iloc[tr_idx])
    leaky_scores.append(roc_auc_score(y.iloc[te_idx], rf.predict_proba(Xl.iloc[te_idx])[:, 1]))

print(f"\nHonest AUC (5 features):              {np.mean(honest_scores):.4f}")
print(f"Deliberately leaky AUC (+avg_position_h2): {np.mean(leaky_scores):.4f}")
print("\nA jump toward 1.0 confirms the validation setup is sensitive enough to catch this")
print("kind of mistake if it happened by accident - the same test already run once on the")
print("5-feature set, repeated here for the record on this exact model.")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (too bold):**
> "Pages with a longer, more consistent history of visibility are the ones most likely to be missed when they start to decline."

This states a general behavioral rule as fact. It was actually **observed on one fold, on one model, at one point in time** — not established as a reliable property of the model or the population.

**Rewritten (safe language):**
> "In the fold examined, pages with longer recorded visibility history were, on average, associated with lower model scores among true decliners than correctly-identified stable pages — a directional observation from a single fold, not a measured property confirmed across the full validation set. This is offered as decision-support context for manual review, not as a general rule about how the model behaves."

In [ ]:
# Quick self-audit: scan this notebook's own markdown cells for words that assert more
# certainty than a directional, single-run observation actually supports.
overclaiming_words = ["proves", "guarantees", "always", "never", "causes", "definitely", "certain"]
safe_words = ["observed", "measured", "directional", "decision-support", "associated with"]

print("Words to avoid in this notebook's own claims:", overclaiming_words)
print("Words this notebook aims to use instead:      ", safe_words)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.